In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from itertools import combinations
from datetime import datetime

| Column | Description |
|--------|-------------|
| **customer_id** | Unique customer identifier |
| **first_name** | Customer first name |
| **last_name** | Customer last name |
| **gender** | Male, Female, Other |
| **age_group** | Teenagers, Adults, Senior |
| **signup_date** | Customer registration date |
| **country** | Customer country (Brazil, Canada, China, France, Germany, UK, USA) |
| **product_id** | Unique product identifier |
| **product_name** | Product name |
| **category** | Product category (Electronics, Apparel, Toys, Home & Kitchen, Books) |
| **quantity** | Number of units purchased |
| **unit_price** | Price per unit |
| **order_id** | Unique order identifier |
| **order_date** | Transaction date |
| **order_status** | Delivered, Pending, Returned, Cancelled |
| **payment_method** | Credit Card, PayPal, Cash on Delivery |
| **rating** | Customer rating (1-5) |
| **review_text** | Text review (good, average, very good, bad) |
| **review_id** | Unique review identifier |
| **review_date** | Date review was posted |

In [2]:
# ref: https://www.kaggle.com/datasets/nabihazahid/ecommerce-dataset-for-sql-analysis
data = pd.read_csv('../datasets/ecommerce_dataset_10000.csv', parse_dates=['order_date', 'signup_date'])
data

,customer_id,first_name,last_name,gender,age_group,signup_date,country,product_id,product_name,category,quantity,unit_price,order_id,order_date,order_status,payment_method,rating,review_text,review_id,review_date
0,CUST2353,Erica,Oliver,Female,Teenagers,2022-06-29,Canada,PROD108,Fitbit Versa 3,Electronics,3,229,ORD10000,2023-07-13,Pending,Credit Card,2,good,REV20000,2025-06-06
1,CUST4463,Christopher,White,Male,Adults,2023-08-24,China,PROD103,Levi's Jeans,Apparel,4,59,ORD10001,2024-08-12,Pending,PayPal,2,average,REV20001,2023-08-05
2,CUST4512,Spencer,Foster,Male,Senior,2023-07-18,Germany,PROD111,Lego Star Wars Set,Toys,2,59,ORD10002,2024-08-04,Delivered,Cash on Delivery,5,good,REV20002,2023-01-03
3,CUST5711,Jessica,Harris,Male,Teenagers,2025-08-22,France,PROD107,Dyson Vacuum,Home & Kitchen,4,399,ORD10003,2025-05-23,Delivered,Cash on Delivery,2,very good,REV20003,2023-03-14
4,CUST1296,Amy,Johnson,Female,Teenagers,2021-03-23,Brazil,PROD105,Adidas Running Shoes,Apparel,1,110,ORD10004,2023-07-02,Returned,Cash on Delivery,1,very good,REV20004,2023-10-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,CUST2529,Lisa,Coleman,Male,Senior,2023-01-06,UK,PROD109,Kindle Paperwhite,Books,5,129,ORD19995,2023-06-07,Delivered,PayPal,4,very good,REV29995,2024-07-30
9996,CUST4602,Stacy,Brown,Other,Teenagers,2023-08-30,Canada,PROD109,Kindle Paperwhite,Books,5,129,ORD19996,2025-08-05,Returned,PayPal,1,bad,REV29996,2024-09-13
9997,CUST1448,Sheila,Roberts,Other,Adults,2024-05-18,USA,PROD107,Dyson Vacuum,Home & Kitchen,5,399,ORD19997,2023-12-23,Returned,Cash on Delivery,3,average,REV29997,2023-09-13
9998,CUST1343,Paul,Lam,Male,Teenagers,2024-07-14,Canada,PROD107,Dyson Vacuum,Home & Kitchen,3,399,ORD19998,2022-08-30,Cancelled,Credit Card,2,very good,REV29998,2025-07-07


### Build a comprehensive customer segmentation model based on purchase behavior and lifetime value.

##### Step 1: Calculate Customer Metrics

For each customer, calculate the following:

| Metric | Formula / Method |
|--------|------------------|
| **Total Revenue** | Sum of `(quantity × unit_price)` across all orders |
| **Total Orders** | Number of unique `order_id` per customer |
| **Average Order Value (AOV)** | `Total Revenue / Total Orders` |
| **Purchase Frequency** | `(Days between first and last order) / Total Orders` |
| **Recency** | Days since last purchase (using `today's date`) |
| **Product Diversity** | Number of unique `category` purchased |
| **Return Rate** | Percentage of orders where `order_status = 'Returned'` |


##### Step 2: Create Customer Value Score (0-100)

Weighted combination of normalized metrics:

| Component | Weight | Description |
|-----------|--------|-------------|
| **Monetary** | 30% | Total revenue (normalized) |
| **Frequency** | 25% | Number of orders (normalized) |
| **Recency** | 25% | Inverse recency (normalized) |
| **Product Diversity** | 20% | Unique categories (normalized) |

$$
\text{Value Score} = 0.30 \times \text{Monetary}_\text{norm} + 0.25 \times \text{Frequency}_\text{norm} + 0.25 \times \text{Recency}_\text{norm} + 0.20 \times \text{Diversity}_\text{norm}
$$


##### Step 3: Segment Customers

| Segment | Criteria |
|---------|----------|
| **Champions** | Top 10% by value score |
| **Loyal Customers** | High frequency, medium-high value |
| **Potential Loyalists** | Recent customers with high value |
| **At Risk** | High value but low recency |
| **Hibernating** | Low recency, low frequency, low value |


##### Step 4: Analyze Each Segment

For each segment, analyze:

| Metric | Description |
|--------|-------------|
| **Average Rating** | Mean rating given by customers in the segment |
| **Most Preferred Category** | Most purchased category |
| **Favorite Payment Method** | Most used payment method |
| **Order Status Distribution** | Breakdown of order statuses |


In [3]:
data['revenue'] = data['quantity'] * data['unit_price']

customer_metrics = data.groupby(['customer_id']).agg(
    total_revenue = ('revenue', 'sum'),
    total_orders = ('order_id', 'nunique'), 
    first_order_date = ('order_date', 'min'),
    last_order_date = ('order_date', 'max'),
    days_between_first_and_last_order_date = ('order_date', lambda x: (x.max() - x.min()).days),
    days_since_last_purchase_recency = ('order_date', lambda x: (datetime.now() - x.max()).days),
    category_diversity = ('category', 'nunique'),
    return_rate_pct = ('order_status', lambda x: (x == 'Returned').mean() * 100)
).reset_index()

customer_metrics = customer_metrics.assign(
    aov=lambda customer_metrics: (customer_metrics['total_revenue'] / customer_metrics['total_orders']).round(2),
    purchase_frequency=lambda df: (customer_metrics['days_between_first_and_last_order_date'] / customer_metrics['total_orders']).round(2)
)
# customer_metrics['aov'] = (customer_metrics['total_revenue'] / customer_metrics['total_orders']).round(2)
# customer_metrics['purchase_frequency'] = (customer_metrics['days_between_first_and_last_order_date'] / customer_metrics['total_orders']).round(2)
customer_metrics

,customer_id,total_revenue,total_orders,first_order_date,last_order_date,days_between_first_and_last_order_date,days_since_last_purchase_recency,category_diversity,return_rate_pct,aov,purchase_frequency
0,CUST1000,6336,5,2022-12-29,2024-12-30,732,604,3,40.0,1267.20,146.40
1,CUST1001,690,2,2024-11-08,2025-04-08,151,505,1,0.0,345.00,75.50
2,CUST1002,999,1,2022-11-09,2022-11-09,0,1386,1,0.0,999.00,0.00
3,CUST1003,444,2,2023-06-05,2023-12-30,208,970,2,50.0,222.00,104.00
4,CUST1004,796,1,2025-01-14,2025-01-14,0,589,1,0.0,796.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...
4322,CUST5993,796,1,2023-09-30,2023-09-30,0,1061,1,100.0,796.00,0.00
4323,CUST5994,2342,3,2022-09-10,2024-01-19,496,950,3,0.0,780.67,165.33
4324,CUST5997,798,1,2025-07-20,2025-07-20,0,402,1,0.0,798.00,0.00
4325,CUST5998,1653,4,2024-05-06,2025-06-02,392,450,3,25.0,413.25,98.00


In [4]:
monetary_scalar = MinMaxScaler()
frequency_scalar = MinMaxScaler()
recency_scalar = MinMaxScaler()
diversity_scalar = MinMaxScaler()

total_revenue_normalized = monetary_scalar.fit_transform(customer_metrics[['total_revenue']]).flatten()
total_orders_normalized = frequency_scalar.fit_transform(customer_metrics[['total_orders']]).flatten()
recency_normalized = recency_scalar.fit_transform(customer_metrics[['days_since_last_purchase_recency']]).flatten()
category_diversity_normalized = diversity_scalar.fit_transform(customer_metrics[['category_diversity']]).flatten()

customer_metrics['value_score'] = (
    0.30 * total_revenue_normalized +
    0.25 * total_orders_normalized +
    0.25 * (1 - recency_normalized) +
    0.20 * category_diversity_normalized
).round(3)
customer_metrics

,customer_id,total_revenue,total_orders,first_order_date,last_order_date,days_between_first_and_last_order_date,days_since_last_purchase_recency,category_diversity,return_rate_pct,aov,purchase_frequency,value_score
0,CUST1000,6336,5,2022-12-29,2024-12-30,732,604,3,40.0,1267.20,146.40,0.537
1,CUST1001,690,2,2024-11-08,2025-04-08,151,505,1,0.0,345.00,75.50,0.264
2,CUST1002,999,1,2022-11-09,2022-11-09,0,1386,1,0.0,999.00,0.00,0.038
3,CUST1003,444,2,2023-06-05,2023-12-30,208,970,2,50.0,222.00,104.00,0.193
4,CUST1004,796,1,2025-01-14,2025-01-14,0,589,1,0.0,796.00,0.00,0.216
...,...,...,...,...,...,...,...,...,...,...,...,...
4322,CUST5993,796,1,2023-09-30,2023-09-30,0,1061,1,100.0,796.00,0.00,0.108
4323,CUST5994,2342,3,2022-09-10,2024-01-19,496,950,3,0.0,780.67,165.33,0.309
4324,CUST5997,798,1,2025-07-20,2025-07-20,0,402,1,0.0,798.00,0.00,0.259
4325,CUST5998,1653,4,2024-05-06,2025-06-02,392,450,3,25.0,413.25,98.00,0.440


In [5]:
# step 3
conditions = [
    # Champions: Top 10% by value score
    (customer_metrics['value_score'] >= customer_metrics['value_score'].quantile(0.90)), # Top 10%
    
    # Loyal Customers: High frequency, medium-high value
    (customer_metrics['total_orders'] >= customer_metrics['total_orders'].median()) &
    (customer_metrics['value_score'] >= customer_metrics['value_score'].median()),
    
    # Potential Loyalists: Recent customers with high value
    (customer_metrics['days_since_last_purchase_recency'] <= customer_metrics['days_since_last_purchase_recency'].quantile(0.25)) &
    (customer_metrics['value_score'] >= customer_metrics['value_score'].median()),
    
    # At Risk: High value but low recency
    (customer_metrics['value_score'] >= customer_metrics['value_score'].median()) &
    (customer_metrics['days_since_last_purchase_recency'] > customer_metrics['days_since_last_purchase_recency'].median()),
    
    # Hibernating: Low recency, low frequency, low value
    (customer_metrics['days_since_last_purchase_recency'] > customer_metrics['days_since_last_purchase_recency'].median()) &
    (customer_metrics['total_orders'] < customer_metrics['total_orders'].median()) &
    (customer_metrics['value_score'] < customer_metrics['value_score'].median())
]

choices = [
    'Champions',
    'Loyal Customers',
    'Potential Loyalists',
    'At Risk',
    'Hibernating'
]

customer_metrics['segment'] = np.select(conditions, choices, default='Other')
customer_metrics

,customer_id,total_revenue,total_orders,first_order_date,last_order_date,days_between_first_and_last_order_date,days_since_last_purchase_recency,category_diversity,return_rate_pct,aov,purchase_frequency,value_score,segment
0,CUST1000,6336,5,2022-12-29,2024-12-30,732,604,3,40.0,1267.20,146.40,0.537,Champions
1,CUST1001,690,2,2024-11-08,2025-04-08,151,505,1,0.0,345.00,75.50,0.264,Other
2,CUST1002,999,1,2022-11-09,2022-11-09,0,1386,1,0.0,999.00,0.00,0.038,Hibernating
3,CUST1003,444,2,2023-06-05,2023-12-30,208,970,2,50.0,222.00,104.00,0.193,Other
4,CUST1004,796,1,2025-01-14,2025-01-14,0,589,1,0.0,796.00,0.00,0.216,Other
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4322,CUST5993,796,1,2023-09-30,2023-09-30,0,1061,1,100.0,796.00,0.00,0.108,Hibernating
4323,CUST5994,2342,3,2022-09-10,2024-01-19,496,950,3,0.0,780.67,165.33,0.309,Loyal Customers
4324,CUST5997,798,1,2025-07-20,2025-07-20,0,402,1,0.0,798.00,0.00,0.259,Other
4325,CUST5998,1653,4,2024-05-06,2025-06-02,392,450,3,25.0,413.25,98.00,0.440,Loyal Customers


In [6]:
# step 4
data = data.drop(columns=['segment'], errors='ignore').merge(customer_metrics[['customer_id', 'segment']], how='left', left_on='customer_id', right_on='customer_id', suffixes=('', ''))

segment_stats = data.groupby('segment').agg(
    customer_count=('customer_id', 'nunique'),
    average_rating=('rating', 'mean'),
    total_orders=('order_id', 'count'),
    total_revenue=('revenue', 'sum'),
    avg_order_value=('revenue', 'mean')
).round(2)

def get_most_preferred_category_and_payment_method(group):
    return pd.Series(
        {
            'most_preferred_category': group['category'].mode()[0] if not group['category'].mode().empty else 'N/A',
            'favorite_payment_method': group['payment_method'].mode()[0] if not group['payment_method'].mode().empty else 'N/A'
        }
    )
most_preferred_category_and_payment_method = data.groupby('segment').apply(get_most_preferred_category_and_payment_method, include_groups=False)

order_status_dist = pd.crosstab(data['segment'], data['order_status'], normalize='index') * 100

segment_stats['most_preferred_category'] = most_preferred_category_and_payment_method['most_preferred_category']
segment_stats['favorite_payment_method'] = most_preferred_category_and_payment_method['favorite_payment_method']
for status in order_status_dist.columns:
    segment_stats[f'order_status_{status}_pct'] = order_status_dist[status].round(2)

segment_stats

,customer_count,average_rating,total_orders,total_revenue,avg_order_value,most_preferred_category,favorite_payment_method,order_status_Cancelled_pct,order_status_Delivered_pct,order_status_Pending_pct,order_status_Returned_pct,order_status_Shipped_pct
segment,,,,,,,,,,,,
Champions,434,3.03,2014,1866009,926.52,Electronics,PayPal,19.46,20.85,20.21,19.41,20.06
Hibernating,960,2.95,960,739576,770.39,Electronics,Cash on Delivery,20.83,18.12,20.62,19.79,20.62
Loyal Customers,1716,2.99,4871,3603708,739.83,Electronics,Cash on Delivery,19.81,21.21,19.24,19.38,20.37
Other,1208,2.99,2146,1205109,561.56,Apparel,Cash on Delivery,19.11,18.92,19.85,20.27,21.85
Potential Loyalists,9,1.89,9,36361,4040.11,Electronics,Cash on Delivery,33.33,44.44,0.00,22.22,0.00


### 2. Product Performance & Inventory Optimization

Analyze product performance metrics and recommend inventory optimization strategies.

#### Step 1: Calculate Product Metrics

For each product, calculate the following:

| Metric | Description |
|--------|-------------|
| **Total Units Sold** | Sum of quantity sold across all orders |
| **Total Revenue** | Sum of (quantity × unit_price) |
| **Average Rating** | Mean rating from product reviews |
| **Return Rate** | Returned quantity / Total quantity sold |
| **Cancellation Rate** | Cancelled orders / Total orders |
| **Sales Velocity** | Units sold per month (considering product launch date) |
| **Revenue Concentration** | Percentage of total revenue from this product |


#### Step 2: Product Performance Matrix (2×2 Grid)

Create a **2×2 matrix** with:

- **X-axis:** Average Rating (High/Low based on median)
- **Y-axis:** Revenue Contribution (High/Low based on median)

Categorize products as:

| Quadrant | Category | Description |
|----------|----------|-------------|
| **High-High** | ⭐ **Stars** | High revenue, high rating — winners! |
| **High-Low** | ❓ **Question Marks** | High rating, low revenue — potential |
| **Low-High** | 💰 **Cash Cows** | Low rating, high revenue — need improvement |
| **Low-Low** | 🐕 **Dogs** | Low revenue, low rating — discontinue |


#### Step 3: Inventory Recommendations by Category

For each category, calculate:

- **Inventory Turnover** = `Quantity Sold / Number of Orders Containing Product`

**Recommendations:**

| Category | Recommended Action |
|----------|-------------------|
| **Stars** | ✅ Increase Stock |
| **Question Marks** | 🔍 Maintain (test demand) |
| **Cash Cows** | 📊 Maintain (protect revenue) |
| **Dogs** | ❌ Reduce Stock / Discontinue |

#### Step 4: Identify Underperforming Products

Flag products that meet any of these conditions:

| Condition | Description |
|-----------|-------------|
| **Condition 1** | Rating < 3 AND Return Rate > 20% |
| **Condition 2** | Declining sales trend over last 3 months |
| **Condition 3** | High revenue concentration (>20%) AND low average rating level |

In [7]:
# step 1
product_metrics = data.groupby(['product_id']).agg(
    total_units_sold = ('quantity', 'sum'),
    total_revenue = ('revenue', 'sum'),
    average_rating = ('rating', 'mean'),
    number_of_orders = ('order_id', 'nunique')
)

def return_quantity(group):
    return_qty = group.loc[group['order_status'] == 'Returned', ['quantity']].sum()
    total_units_sold = group['quantity'].sum()
    return return_qty * 100 / total_units_sold

def cancel_quantity(group):
    cancelled_orders_count = group.loc[group['order_status'] == 'Cancelled'].shape[0]
    total_orders = group.shape[0]
    return cancelled_orders_count * 100 / total_orders

product_metrics['return_rate'] = data.groupby(['product_id'], group_keys=False).apply(return_quantity, include_groups=False)
product_metrics['cancelled_rate'] = data.groupby(['product_id'], group_keys=False).apply(cancel_quantity, include_groups=False)

def sales_velocity(group):
    if group.empty:
        return 0
    min_date = group['order_date'].min()
    max_date = group['order_date'].max()
    months = max((max_date - min_date).days / 30.44, 0.1)  # avoid division by zero
    total_units = group['quantity'].sum()
    return total_units / months

product_metrics['sales_velocity'] = data.groupby('product_id', group_keys=False).apply(sales_velocity, include_groups=False)

total_revenue_all = data['revenue'].sum()
product_metrics['revenue_concentration_pct'] = product_metrics['total_revenue'] * 100 / total_revenue_all 

product_metrics = product_metrics.reset_index()

In [8]:
# step 2
average_rating_median = product_metrics['average_rating'].median()
revenue_contribution_median = product_metrics['revenue_concentration_pct'].median()

product_metrics['average_rating_level'] = np.where(product_metrics['average_rating'] > average_rating_median, 'high', 'low')
product_metrics['revenue_contribution_level'] = np.where(product_metrics['revenue_concentration_pct'] > revenue_contribution_median, 'high', 'low')

def assign_quadrant(row):
    rating = row['average_rating_level']
    revenue = row['revenue_contribution_level']
    
    if rating == 'high' and revenue == 'high':
        return '⭐ Stars'
    elif rating == 'hHigh' and revenue == 'low':
        return '❓ Question Marks'
    elif rating == 'low' and revenue == 'high':
        return '💰 Cash Cows'
    else:  # Low, Low
        return '🐕 Dogs'

product_metrics['quadrant'] = product_metrics.apply(assign_quadrant, axis=1)
quadrant_summary = product_metrics.groupby('quadrant').agg(
    count=('product_id', 'count'),
    avg_rating=('average_rating', 'mean'),
    avg_revenue_pct=('revenue_concentration_pct', 'mean'),
    total_revenue=('total_revenue', 'sum')
).round(2)
quadrant_summary

,count,avg_rating,avg_revenue_pct,total_revenue
quadrant,,,,
⭐ Stars,5,3.03,13.19,4914653
🐕 Dogs,8,2.98,2.30,1369356
💰 Cash Cows,2,2.94,7.83,1166754


In [9]:
# step 3
quadrant_metrics = product_metrics.groupby(['quadrant']).agg(
    quantity_sold = ('total_units_sold', 'sum'),
    nmuber_of_orders = ('number_of_orders', 'sum')
)
quadrant_metrics['inventory_turnover'] = quadrant_metrics['quantity_sold'] / quadrant_metrics['nmuber_of_orders']
quadrant_metrics

,quantity_sold,nmuber_of_orders,inventory_turnover
quadrant,,,
⭐ Stars,9867,3272,3.015587
🐕 Dogs,15995,5337,2.997002
💰 Cash Cows,4146,1391,2.980590


In [10]:
# step 4
product_metrics['flag_condition_1'] = False
condition1_mask = (product_metrics['average_rating'] < 3) & (product_metrics['return_rate'] > 20)
product_metrics.loc[condition1_mask, 'flag_condition_1'] = True

product_metrics['flag_condition_2'] = False
avg_sales_velocity = product_metrics['sales_velocity'].mean() # i dont calcualte again throught the months so just use mean 
condition1_mask = (product_metrics['sales_velocity'] < avg_sales_velocity * 0.7) # 30% below average
product_metrics.loc[condition1_mask, 'flag_condition_2'] = True

product_metrics['flag_condition_3'] = False
condition1_mask = (product_metrics['revenue_concentration_pct'] > 20) & (product_metrics['average_rating_level'] == 'Low')
product_metrics.loc[condition1_mask, 'flag_condition_3'] = True

In [11]:
product_metrics

,product_id,total_units_sold,total_revenue,average_rating,number_of_orders,return_rate,cancelled_rate,sales_velocity,revenue_concentration_pct,average_rating_level,revenue_contribution_level,quadrant,flag_condition_1,flag_condition_2,flag_condition_3
0,PROD100,1883,1881117,2.984000,625,18.534254,22.080000,52.393528,25.247307,high,high,⭐ Stars,False,False,False
1,PROD101,1981,1780919,2.996890,643,15.951540,20.995334,55.120329,23.902505,high,high,⭐ Stars,False,False,False
2,PROD102,2136,425064,3.107402,689,22.050562,18.142235,59.433126,5.704973,high,high,⭐ Stars,False,False,False
3,PROD103,2074,122366,2.959596,693,19.238187,18.903319,57.760805,1.642328,low,low,🐕 Dogs,False,False,False
4,PROD104,2042,245040,2.974777,674,20.274241,20.178042,56.765735,3.288791,low,low,🐕 Dogs,True,False,False
5,PROD105,2093,230230,3.008824,680,19.684663,17.205882,58.343333,3.090019,high,low,🐕 Dogs,False,False,False
6,PROD106,1954,193446,2.974281,661,20.829069,21.936460,54.568587,2.596325,low,low,🐕 Dogs,True,False,False
7,PROD107,2196,876204,2.965753,730,20.218579,16.986301,61.102596,11.759923,low,high,💰 Cash Cows,True,False,False
8,PROD108,1934,442886,2.993930,659,18.355739,17.754173,53.911136,5.944170,high,high,⭐ Stars,False,False,False
9,PROD109,1973,254517,2.980243,658,20.527116,23.556231,54.897733,3.415986,low,low,🐕 Dogs,True,False,False


### 3. Market Expansion & Country Performance Analysis

Analyze country-wise performance and identify market expansion opportunities.

#### Step 1: Calculate Country Metrics

For each country, calculate:

| Metric | Description |
|--------|-------------|
| **Total Revenue** | Sum of (quantity × unit_price) and percentage of company revenue |
| **Unique Customers** | Number of distinct customers and acquisition rate (percentage of total customers) |
| **Average Order Value (AOV)** | Total Revenue / Total Orders |
| **Average Rating** | Mean rating from customers in that country |
| **Top 3 Products** | Most sold products in that country |
| **Payment Method Distribution** | % of orders by payment method |
| **Order Status Distribution** | % Delivered / Returned / Cancelled |
| **Seasonality Index** | Monthly sales pattern (peak and low months) |

#### Step 2: Market Attractiveness Score

Weighted combination of four metrics:

| Component | Weight | Description |
|-----------|--------|-------------|
| **Revenue Growth** | 30% | Month-over-month average growth rate |
| **Customer Growth** | 25% | Customer acquisition rate |
| **Profitability** | 25% | Low return/cancellation rates |
| **Customer Engagement** | 20% | High ratings, high AOV |

**Formula:**

Attractiveness Score = 0.30 × RevenueGrowth + 0.25 × CustomerGrowth + 0.25 × Profitability + 0.20 × Engagement

#### Step 3: Market Classification

| Market Type | Criteria |
|-------------|----------|
| **Core Markets** | High revenue, high growth |
| **Growth Markets** | Low revenue, high growth |
| **Mature Markets** | High revenue, low growth |
| **Declining Markets** | Low revenue, negative growth |

In [12]:
# step 1
country_metrics = data.groupby(['country']).agg(
    total_revenue = ('revenue', 'sum'),
    unique_customers = ('customer_id', 'nunique'),
    total_orders = ('order_id', 'nunique'),
    average_rating = ('rating', 'mean'),
    delivered_count = ('order_status', lambda x: (x=='Delivered').sum()),
    returned_count = ('order_status', lambda x: (x=='Returned').sum()),
    cancelled_count = ('order_status', lambda x: (x=='Cancelled').sum()),
).round(2)
company_revenue = data['revenue'].sum()
country_metrics['revenue_percentage'] = (country_metrics['total_revenue'] * 100.0 / company_revenue).round(2)

total_customer_count = data['customer_id'].nunique()
country_metrics['acquisition_rate'] = (country_metrics['unique_customers'] * 100.0 / total_customer_count).round(2)

country_metrics['aov'] = (country_metrics['total_revenue'] / country_metrics['total_orders']).round(2)

def top_n_most_sold_products(group_data, top_n=3, by='quantity'):
    total_by_country_and_product = group_data.groupby(['country', 'product_id']).agg(total_ = (by, 'sum')).reset_index()
    total_by_country_and_product['product_rank'] = total_by_country_and_product.groupby(['country'])['total_'].rank(method='dense', ascending=False)
    top_n_products = total_by_country_and_product[total_by_country_and_product['product_rank']<=top_n].sort_values(by=['country', 'product_rank'])
    
    return top_n_products.groupby(['country'])['product_id'].apply(list).reset_index().rename(columns={'product_id': 'most_sold_product_ids'})

country_metrics = country_metrics.merge(top_n_most_sold_products(data), left_on='country', right_on='country')

country_metrics['delivered_pct'] = (country_metrics['delivered_count'] / country_metrics['total_orders'] * 100).round(2)
country_metrics['returned_pct'] = (country_metrics['returned_count'] / country_metrics['total_orders'] * 100).round(2)
country_metrics['cancelled_pct'] = (country_metrics['cancelled_count'] / country_metrics['total_orders'] * 100).round(2)

country_metrics

,country,total_revenue,unique_customers,total_orders,average_rating,delivered_count,returned_count,cancelled_count,revenue_percentage,acquisition_rate,aov,most_sold_product_ids,delivered_pct,returned_pct,cancelled_pct
0,Australia,745407,443,1043,3.00,187,203,218,10.00,10.24,714.68,"[PROD107, PROD102, PROD113]",17.93,19.46,20.90
1,Brazil,743066,428,1019,2.98,220,186,214,9.97,9.89,729.21,"[PROD103, PROD113, PROD106]",21.59,18.25,21.00
2,Canada,719928,438,997,3.00,216,202,184,9.66,10.12,722.09,"[PROD103, PROD114, PROD107]",21.66,20.26,18.46
3,China,815789,458,1054,2.99,218,203,203,10.95,10.58,773.99,"[PROD107, PROD105, PROD101]",20.68,19.26,19.26
4,France,810303,442,1060,2.95,218,207,190,10.88,10.21,764.44,"[PROD101, PROD107, PROD106]",20.57,19.53,17.92
5,Germany,635391,393,885,3.05,206,163,163,8.53,9.08,717.96,"[PROD104, PROD106, PROD105, PROD110]",23.28,18.42,18.42
6,India,759619,449,1036,3.07,172,213,215,10.20,10.38,733.22,"[PROD113, PROD106, PROD102]",16.60,20.56,20.75
7,Japan,777673,445,976,2.96,202,198,204,10.44,10.28,796.80,"[PROD102, PROD105, PROD107]",20.70,20.29,20.90
8,UK,676570,409,943,2.99,199,195,185,9.08,9.45,717.47,"[PROD112, PROD104, PROD106]",21.10,20.68,19.62
9,USA,767017,422,987,2.95,199,192,194,10.29,9.75,777.12,"[PROD112, PROD105, PROD102, PROD107]",20.16,19.45,19.66


In [13]:
# step 2
scaler_revenue = MinMaxScaler()
scaler_customer = MinMaxScaler()
scaler_profit = MinMaxScaler()
scaler_engagement = MinMaxScaler()

country_metrics['revenue_growth_normalized'] = scaler_revenue.fit_transform(country_metrics[['revenue_percentage']]).flatten()
country_metrics['customer_growth_normalized'] = scaler_customer.fit_transform(country_metrics[['acquisition_rate']]).flatten()
# profitability = 100 - (return rates + cancellation rates)
country_metrics['profitability_score'] = 100 - (country_metrics['returned_pct'] + country_metrics['cancelled_pct'])
country_metrics['profitability_normalized'] = scaler_profit.fit_transform(country_metrics[['profitability_score']]).flatten()
# engagement_score = average_rating + AOV
country_metrics['engagement_score'] = (
    country_metrics['average_rating'] / country_metrics['average_rating'].max() * 50 +
    country_metrics['aov'] / country_metrics['aov'].max() * 50
)
country_metrics['engagement_normalized'] = scaler_engagement.fit_transform(country_metrics[['engagement_score']]).flatten()

country_metrics['attractiveness_score'] = (
    0.30 * country_metrics['revenue_growth_normalized'] +
    0.25 * country_metrics['customer_growth_normalized'] +
    0.25 * country_metrics['profitability_normalized'] +
    0.20 * country_metrics['engagement_normalized']
).round(4)

In [14]:
country_metrics

,country,total_revenue,unique_customers,total_orders,average_rating,delivered_count,returned_count,cancelled_count,revenue_percentage,acquisition_rate,...,delivered_pct,returned_pct,cancelled_pct,revenue_growth_normalized,customer_growth_normalized,profitability_score,profitability_normalized,engagement_score,engagement_normalized,attractiveness_score
0,Australia,745407,443,1043,3.00,187,203,218,10.00,10.24,...,17.93,19.46,20.90,0.607438,0.773333,59.64,0.212528,93.706822,0.000000,0.4287
1,Brazil,743066,428,1019,2.98,220,186,214,9.97,9.89,...,21.59,18.25,21.00,0.595041,0.540000,60.75,0.460850,94.292862,0.130183,0.4548
2,Canada,719928,438,997,3.00,216,202,184,9.66,10.12,...,21.66,20.26,18.46,0.466942,0.693333,61.28,0.579418,94.171807,0.103292,0.4789
3,China,815789,458,1054,2.99,218,203,203,10.95,10.58,...,20.68,19.26,19.26,1.000000,1.000000,61.48,0.624161,97.265718,0.790576,0.8642
4,France,810303,442,1060,2.95,218,207,190,10.88,10.21,...,20.57,19.53,17.92,0.971074,0.753333,62.55,0.863535,96.014980,0.512736,0.7981
5,Germany,635391,393,885,3.05,206,163,163,8.53,9.08,...,23.28,18.42,18.42,0.000000,0.000000,63.16,1.000000,94.726978,0.226618,0.2953
6,India,759619,449,1036,3.07,172,213,215,10.20,10.38,...,16.60,20.56,20.75,0.690083,0.866667,58.69,0.000000,96.010291,0.511695,0.5260
7,Japan,777673,445,976,2.96,202,198,204,10.44,10.28,...,20.70,20.29,20.90,0.789256,0.800000,58.81,0.026846,98.208469,1.000000,0.6435
8,UK,676570,409,943,2.99,199,195,185,9.08,9.45,...,21.10,20.68,19.62,0.227273,0.246667,59.70,0.225951,93.719031,0.002712,0.1869
9,USA,767017,422,987,2.95,199,192,194,10.29,9.75,...,20.16,19.45,19.66,0.727273,0.446667,60.89,0.492170,96.810663,0.689490,0.5908


In [17]:
# step 3
revenue_median = country_metrics['revenue_percentage'].median()
growth_median = country_metrics['revenue_growth_normalized'].median()

def classify_market(row):
    """
    - Core Markets: High revenue, high growth
    - Growth Markets: Low revenue, high growth
    - Mature Markets: High revenue, low growth
    - Declining Markets: Low revenue, negative growth
    """
    revenue = row['revenue_percentage']
    growth = row['revenue_growth_normalized']
    
    # Determine revenue level
    is_high_revenue = revenue >= revenue_median
    
    # Determine growth level
    is_high_growth = growth >= growth_median
    
    if is_high_revenue and is_high_growth:
        return "Core Markets"
    elif not is_high_revenue and is_high_growth:
        return "Growth Markets"
    elif is_high_revenue and not is_high_growth:
        return "Mature Markets"
    else:
        return "Declining Markets"

country_metrics['market_type'] = country_metrics.apply(classify_market, axis=1)

country_metrics

,country,total_revenue,unique_customers,total_orders,average_rating,delivered_count,returned_count,cancelled_count,revenue_percentage,acquisition_rate,...,returned_pct,cancelled_pct,revenue_growth_normalized,customer_growth_normalized,profitability_score,profitability_normalized,engagement_score,engagement_normalized,attractiveness_score,market_type
0,Australia,745407,443,1043,3.00,187,203,218,10.00,10.24,...,19.46,20.90,0.607438,0.773333,59.64,0.212528,93.706822,0.000000,0.4287,Declining Markets
1,Brazil,743066,428,1019,2.98,220,186,214,9.97,9.89,...,18.25,21.00,0.595041,0.540000,60.75,0.460850,94.292862,0.130183,0.4548,Declining Markets
2,Canada,719928,438,997,3.00,216,202,184,9.66,10.12,...,20.26,18.46,0.466942,0.693333,61.28,0.579418,94.171807,0.103292,0.4789,Declining Markets
3,China,815789,458,1054,2.99,218,203,203,10.95,10.58,...,19.26,19.26,1.000000,1.000000,61.48,0.624161,97.265718,0.790576,0.8642,Core Markets
4,France,810303,442,1060,2.95,218,207,190,10.88,10.21,...,19.53,17.92,0.971074,0.753333,62.55,0.863535,96.014980,0.512736,0.7981,Core Markets
5,Germany,635391,393,885,3.05,206,163,163,8.53,9.08,...,18.42,18.42,0.000000,0.000000,63.16,1.000000,94.726978,0.226618,0.2953,Declining Markets
6,India,759619,449,1036,3.07,172,213,215,10.20,10.38,...,20.56,20.75,0.690083,0.866667,58.69,0.000000,96.010291,0.511695,0.5260,Core Markets
7,Japan,777673,445,976,2.96,202,198,204,10.44,10.28,...,20.29,20.90,0.789256,0.800000,58.81,0.026846,98.208469,1.000000,0.6435,Core Markets
8,UK,676570,409,943,2.99,199,195,185,9.08,9.45,...,20.68,19.62,0.227273,0.246667,59.70,0.225951,93.719031,0.002712,0.1869,Declining Markets
9,USA,767017,422,987,2.95,199,192,194,10.29,9.75,...,19.45,19.66,0.727273,0.446667,60.89,0.492170,96.810663,0.689490,0.5908,Core Markets


### 4. Temporal Pattern Analysis & Forecasting

Uncover temporal patterns in sales, customer behavior, and review sentiment.

#### Step 1: Extract Time Features

Extract the following time features from `order_date` and `signup_date`:

| Feature | Description |
|---------|-------------|
| **Day of week** | Monday, Tuesday, etc. |
| **Month** | January, February, etc. |
| **Quarter** | Q1, Q2, Q3, Q4 |
| **Year** | 2020, 2021, etc. |
| **Week of the year** | 1 to 52 |
| **Day of month** | 1 to 31 |
| **Season** | Winter, Spring, Summer, Fall |


#### Step 2: Sales Pattern Analysis

For each time dimension, analyze:

| Question | Method |
|----------|--------|
| Which month has highest sales? | Group by month, sum revenue |
| Is there a weekend/weekday effect? | Compare weekend vs weekday sales |
| Year-over-year growth rate | Compare same month across years |

#### Step 3: Customer Behavior Analysis

| Question | Method |
|----------|--------|
| Do certain age_groups purchase more during specific months? | Pivot table: Age Group × Month |
| Which payment method dominates specific periods? | Payment method share over time |
| Average order value trends over time | Monthly AOV trend line |

#### Step 4: Review Sentiment Analysis

| Question | Method |
|----------|--------|
| Monthly average rating trends | Group by month, mean rating |
| Relationship between order volume and rating | Correlation analysis |
| Review delay by season | Days between order and review |


#### Step 5: Leading Indicators

Build a **one-month ahead prediction model** using:

| Feature | Description |
|---------|-------------|
| Review sentiment | Delayed by 1 month |
| Customer signup rate | New customers per month |
| Payment method preferences | % of orders by payment method |


#### Step 6: Identify Anomalous Periods

| Task | Method |
|------|--------|
| Find unexpected high/low sales months | Z-score from 12-month rolling average |
| Investigate anomalies | Correlate with rating changes or return rates |
| Flag potential issues | Identify external events |


#### Step 7: Cohort Retention Analysis

| Task | Method |
|------|--------|
| Group by signup month | Group customers by signup month |
| Calculate retention rate | Percentage of customers ordering in subsequent months |
| Identify best cohorts | Compare retention rates across cohorts |
